In [1]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [3]:
# data = pd.read_stata(r"datos/ECU_2004m12_BID.dta", convert_categoricals=False) # para bases de stata

data3 = pd.read_stata(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2006/m3/data_orig/ecu06-mar.dta", convert_categoricals=False) # para bases de stata
data6 = pd.read_stata(r"/home/edu/Dropbox/datos/ECU/ECU//ENEMDU/2006/m6/data_orig/ecu06-jun.dta", convert_categoricals=False) # para bases de stata
data9 = pd.read_stata(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2006/m9/data_orig/ecu06-sep.dta", convert_categoricals=False) # para bases de stata
data12 = pd.read_stata(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2006/m12/data_orig/per12_2006.dta", convert_categoricals=False) # para bases de stata

## Revisar los datos

| marzo | junio | septiembre | diciembre |
|-----------|-----------|-----------|-----------|
| area  | area  | area  | area  |
| ciudad  | ciudad  | ciudad  | ciudad  |
| zona  | zona  | zona  | zona  |
| sector  | sector  | sector  | sector  |
| panelm  | panelm  | panelm  | panelm  |
| vivienda  | vivienda  | vivienda  | vivienda  |
| hogar  | hogar  | hogar  | hogar  |
| sexo  | sexo  | sexo  | sexo  |
| edad  | edad  | edad  | edad  |
| pe63  | pe63  | pe63  | pe63  |
| fexp  | fexp  | fexp  | fexp  |
| trabajo  | trabajo  | trabajo  | trabajo  |

En esta encuesta tenemos separadas cuatro diferentes bases para cada trimestre, esto cambia la lógica que habíamos tenido hasta ahora así que de aquí en adelante cambiamos algo del código, mantenemos de acuerdo a las etiquetas de las variables pe63 como la variable de ingreso laboral monetario de la actividad principal asalariada para mantener la concordancia con el resto de los años.

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no trabajando, estas variables las usamos antes para identificar la condición de trabajo para diferentes meses en encuestas anuales o incompletas donde asumíamos que mantenía el mismo salario si estaba ocupado en ese mes, sin mbargo estas variables tenían el problema de no corresponder de forma exacta con el año o mes de la encuesta. Ahora sin embargo podemos cambiar las suposiciones y solamente asumir que si la variable 'trabajando' que pregunta si el individuo trabajó la semana pasada se cumple vamos a asumir que trabajo durante todo el trimestre, de esta manera podemos mejorar las suposiciones de ocupación mensual, mantenemos la idea de que si el individuo trabajo recibe su ingreso laboral reportado.

In [4]:
columnas = pd.Index(['area', 'ciudad', 'zona', 'sector', 'panelm',
            'vivienda', 'hogar', 'sexo', 'edad', 'pe63', 'trabajo',
            'fexp'])

Filtramos solo las columnas de interés para alivar el peso en la memoria

In [5]:
data3 = data3[columnas]
data6 = data6[columnas]
data9 = data9[columnas.intersection(data9.columns)]
data12 = data12[columnas.intersection(data12.columns)]

Creamos una variable de ingreso laboral que es igual al ingreso por asalariado y limpiamos según los valores de ingrl, para mantener ambas variables para cada base consistente

In [6]:
data3['pe63'] = pd.to_numeric(data3['pe63'], errors='coerce')
data3['pe63'] = data3['pe63'].apply(lambda x: np.nan if x > 6400 else x)

data6['pe63'] = pd.to_numeric(data6['pe63'], errors='coerce')
data6['pe63'] = data6['pe63'].apply(lambda x: np.nan if x > 5000 else x)

data9['pe63'] = pd.to_numeric(data9['pe63'], errors='coerce')
data9['pe63'] = data9['pe63'].apply(lambda x: np.nan if x > 5000 else x)

data12['pe63'] = pd.to_numeric(data12['pe63'], errors='coerce')
data12['pe63'] = data12['pe63'].apply(lambda x: np.nan if x > 6000 else x)

In [7]:
data3['ingr'] = data3['pe63']
data6['ingr'] = data6['pe63']
data9['ingr'] = data9['pe63']
data12['ingr'] = data12['pe63']

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados la semana pasada, de acuerdo a la variable 'trabajo'

In [8]:
data3['ingr_t1'] = data3.apply(lambda x: x['ingr'] if x['trabajo'] == 1 else np.nan, axis=1)

data6['ingr_t2'] = data6.apply(lambda x: x['ingr'] if x['trabajo'] == 1 else np.nan, axis=1)

data9['ingr_t3'] = data9.apply(lambda x: x['ingr'] if x['trabajo'] == 1 else np.nan, axis=1)

data12['ingr_t4'] = data12.apply(lambda x: x['ingr'] if x['trabajo'] == 1 else np.nan, axis=1)

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [9]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 2006]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc

In [10]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_base.iterrows()
     }

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [11]:
# Corregimos los códigos para usarlos cómo texto
data3['ciudad'] = data3['ciudad'].apply(str)
data3['ciudad'] = data3['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
data3['ciudad_2'] = data3['ciudad'].apply(lambda x: x[:4])

data6['ciudad'] = data6['ciudad'].apply(str)
data6['ciudad'] = data6['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
data6['ciudad_2'] = data6['ciudad'].apply(lambda x: x[:4])

data9['ciudad'] = data9['ciudad'].apply(str)
data9['ciudad'] = data9['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
data9['ciudad_2'] = data9['ciudad'].apply(lambda x: x[:4])

data12['ciudad'] = data12['ciudad'].apply(str)
data12['ciudad'] = data12['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
data12['ciudad_2'] = data12['ciudad'].apply(lambda x: x[:4])

Diccionario ciudades disponibles

In [12]:
parroquia_dict = {
    '0101': 'Cuenca',
    '0901': 'Guayaquil',
    '0801': 'Esmeraldas',
    '0701': 'Machala',
    '1308': 'Manta',
    '1701': 'Quito',
    '1101': 'Loja',
    '1801': 'Ambato'
}

def get_parroquia(codigo):
    if codigo in parroquia_dict:
        return parroquia_dict[codigo]
    elif codigo[:2] in ['01', '02', '03', '04', '05', '06', '10', '11', '17', '18']:
        return 'Sierra'
    elif codigo[:2] in ['07', '08', '09', '12', '13', '23', '24']:
        return 'Costa'
    else:
        return 'Nacional'

data3['ciudad_asignada'] = data3['ciudad_2'].apply(get_parroquia)
data6['ciudad_asignada'] = data6['ciudad_2'].apply(get_parroquia)
data9['ciudad_asignada'] = data9['ciudad_2'].apply(get_parroquia)
data12['ciudad_asignada'] = data12['ciudad_2'].apply(get_parroquia)

In [13]:
data9['ciudad_asignada'].value_counts()

ciudad_asignada
Costa         4987
Guayaquil     4578
Quito         3850
Sierra        2998
Nacional      2016
Machala       1997
Cuenca        1890
Manta          456
Ambato         425
Esmeraldas     352
Loja           318
Name: count, dtype: int64

### Asignamos el ipc correspondiente según ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = ingr_{dólares}^{i}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Desde 2000 en adelante ya no es necesario utilizar el tipo de cambio debido al cambio de moneda

In [14]:
# Función que asigna valores correspondientes
def asigna_ipc(fila, trimestre):
    return ipc_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila, trimestre):
    return ipc_base_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

In [15]:
data3['ipc_t1'] = data3.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
data3['ipc_base_t1'] = data3.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)

data6['ipc_t2'] = data6.apply(lambda fila: asigna_ipc(fila, 2), axis=1)
data6['ipc_base_t2'] = data6.apply(lambda fila: asigna_ipc_base(fila, 2), axis=1)

data9['ipc_t3'] = data9.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data9['ipc_base_t3'] = data9.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)

data12['ipc_t4'] = data12.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data12['ipc_base_t4'] = data12.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)

In [16]:
# Calculamos el deflactor
data3['def_t1'] = (data3['ipc_base_t1'] / data3['ipc_t1'])
data6['def_t2'] = (data6['ipc_base_t2'] / data6['ipc_t2'])
data9['def_t3'] = (data9['ipc_base_t3'] / data9['ipc_t3'])
data12['def_t4'] = (data12['ipc_base_t4'] / data12['ipc_t4'])

Ingreso promedio en el trimeste

In [17]:
data3['ingr_t1_r'] = data3['ingr_t1'] * data3['def_t1']
data6['ingr_t2_r'] = data6['ingr_t2'] * data6['def_t2']
data9['ingr_t3_r'] = data9['ingr_t3'] * data9['def_t3']
data12['ingr_t4_r'] = data12['ingr_t4'] * data12['def_t4']

In [18]:
print(data3['ingr_t1_r'].mean())
print(data6['ingr_t2_r'].mean())
print(data9['ingr_t3_r'].mean())
print(data12['ingr_t4_r'].mean())

368.38684881554366
363.21263955365623
397.01427438660687
295.2103861887855


## Regiones

In [20]:
# Corregimos los códigos para usarlos cómo texto
data3['ciudad'] = data3['ciudad'].apply(str)
data3['ciudad'] = data3['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data3['ciudad_2'] = data3['ciudad'].apply(lambda x: x[:2])

data6['ciudad'] = data6['ciudad'].apply(str)
data6['ciudad'] = data6['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data6['ciudad_2'] = data6['ciudad'].apply(lambda x: x[:2])

data9['ciudad'] = data9['ciudad'].apply(str)
data9['ciudad'] = data9['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data9['ciudad_2'] = data9['ciudad'].apply(lambda x: x[:2])

data12['ciudad'] = data12['ciudad'].apply(str)
data12['ciudad'] = data12['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data12['ciudad_2'] = data12['ciudad'].apply(lambda x: x[:2])

In [21]:
regiones_dict = {
    'Guayas': '09',
    'Manabí': '13',
    'El Oro': '07',
    'Los Ríos': '12',
    'Pichincha': '17',
    'Azuay': '01',
    'Galápagos': '20',
    'Sierra': ['04', '10', '05', '18', '02', '06', '03', '11'],
    'Costa, Santo Domingo': ['08', '24', '23'],
    'Amazonía': ['14', '15', '16', '19', '21', '22', '90']
}

In [22]:
codigo_region = {}
for region, codes in regiones_dict.items():
    
    if isinstance(codes, list):
        for code in codes:
            codigo_region[code] = region
    
    else:
        codigo_region[codes] = region

# Mapeo de regiones
data3['region'] = data3['ciudad_2'].map(codigo_region)

data6['region'] = data6['ciudad_2'].map(codigo_region)

data9['region'] = data9['ciudad_2'].map(codigo_region)

data12['region'] = data12['ciudad_2'].map(codigo_region)

In [23]:
data3['region'].value_counts()

region
Guayas                  6740
Pichincha               4749
Sierra                  2764
El Oro                  2370
Amazonía                2068
Azuay                   1896
Manabí                  1566
Los Ríos                 893
Costa, Santo Domingo     630
Name: count, dtype: int64

In [24]:
data6['region'].value_counts()

region
Guayas                  6494
Pichincha               4748
Sierra                  2841
El Oro                  2329
Amazonía                1935
Azuay                   1916
Manabí                  1610
Los Ríos                 939
Costa, Santo Domingo     628
Name: count, dtype: int64

In [25]:
data9['region'].value_counts()

region
Guayas                  6808
Pichincha               4679
Sierra                  2827
El Oro                  2427
Amazonía                2016
Azuay                   1975
Manabí                  1577
Los Ríos                 966
Costa, Santo Domingo     592
Name: count, dtype: int64

In [26]:
data12['region'].value_counts()

region
Sierra                  29717
Guayas                   9632
Pichincha                6779
Costa, Santo Domingo     6154
Manabí                   6127
Los Ríos                 5968
El Oro                   5498
Amazonía                 4220
Azuay                    3869
Name: count, dtype: int64

## Calculo ingreso de los hogares

In [27]:
columnas_idef = pd.Index(['area', 'ciudad', 'zona', 'sector', 'vivienda',
       'hogar'])

data3['idef_hogar'] = data3[columnas_idef].astype(str).agg(''.join, axis=1)
data6['idef_hogar'] = data6[columnas_idef].astype(str).agg(''.join, axis=1)
data9['idef_hogar'] = data9[
    columnas_idef.intersection(data9.columns)
    ].astype(str).agg(''.join, axis=1)
data12['idef_hogar'] = data12[
    columnas_idef.intersection(data12.columns)
    ].astype(str).agg(''.join, axis=1)

print(len(data3['idef_hogar'].unique()))
print(len(data6['idef_hogar'].unique()))
print(len(data9['idef_hogar'].unique()))
print(len(data12['idef_hogar'].unique()))

1540
1878
3362
6219


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [28]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [29]:
data3['ingr_t1_h'] = data3.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)
data6['ingr_t2_h'] = data6.groupby('idef_hogar')['ingr_t2_r'].transform(sum_with_na)
data9['ingr_t3_h'] = data9.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)
data12['ingr_t4_h'] = data12.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

In [30]:
print(data3['ingr_t1_h'].mean())
print(data6['ingr_t2_h'].mean())
print(data9['ingr_t3_h'].mean())
print(data12['ingr_t4_h'].mean())

1460.8992610921864
1324.6272169920903
919.5630462621906
1059.2636521902516


## Sacamos edades negativas y mayores a 100 años

In [31]:
print(len(data3))
print(len(data6))
print(len(data9))
print(len(data12))

23676
23440
23867
77964


Transformamos las variables de edad a numericas para evitar problemas

In [32]:
data3['edad'] = pd.to_numeric(data3['edad'], errors='coerce')
data6['edad'] = pd.to_numeric(data6['edad'], errors='coerce')
data9['edad'] = pd.to_numeric(data9['edad'], errors='coerce')
data12['edad'] = pd.to_numeric(data12['edad'], errors='coerce')

In [33]:
data3 = data3.loc[(data3['edad'] >= 0) & (data3['edad'] < 100)]
data6 = data6.loc[(data6['edad'] >= 0) & (data6['edad'] < 100)]
data9 = data9.loc[(data9['edad'] >= 0) & (data9['edad'] < 100)]
data12 = data12.loc[(data12['edad'] >= 0) & (data12['edad'] < 100)]

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [34]:
k = 0.4
s = 0.9

In [35]:
# Si es necesario calcular el número de niños
data3['es_nino'] = data3['edad'] < 10
data3['ninos'] = data3.groupby('idef_hogar')['es_nino'].transform('sum')

data6['es_nino'] = data6['edad'] < 10
data6['ninos'] = data6.groupby('idef_hogar')['es_nino'].transform('sum')

data9['es_nino'] = data9['edad'] < 10
data9['ninos'] = data9.groupby('idef_hogar')['es_nino'].transform('sum')

data12['es_nino'] = data12['edad'] < 10
data12['ninos'] = data12.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data3['es_adulto'] = data3['edad'] > 10
data3['adultos'] = data3.groupby('idef_hogar')['es_adulto'].transform('sum')

data6['es_adulto'] = data6['edad'] > 10
data6['adultos'] = data6.groupby('idef_hogar')['es_adulto'].transform('sum')

data9['es_adulto'] = data9['edad'] > 10
data9['adultos'] = data9.groupby('idef_hogar')['es_adulto'].transform('sum')

data12['es_adulto'] = data12['edad'] > 10
data12['adultos'] = data12.groupby('idef_hogar')['es_adulto'].transform('sum')

In [36]:
data3['escala'] = (data3['adultos'] + k * data3['ninos']) ** s
data6['escala'] = (data6['adultos'] + k * data6['ninos']) ** s
data9['escala'] = (data9['adultos'] + k * data9['ninos']) ** s
data12['escala'] = (data12['adultos'] + k * data12['ninos']) ** s

In [37]:
data3['ingr_t_t1'] = data3['ingr_t1_h'] / data3['escala']
data6['ingr_t_t2'] = data6['ingr_t2_h'] / data6['escala']
data9['ingr_t_t3'] = data9['ingr_t3_h'] / data9['escala']
data12['ingr_t_t4'] = data12['ingr_t4_h'] / data12['escala']

In [38]:
print(data3['ingr_t_t1'].mean())
print(data6['ingr_t_t2'].mean())
print(data9['ingr_t_t3'].mean())
print(data12['ingr_t_t4'].mean())

135.13187019145695
135.16324652143803
158.06982909682287
108.27607754721049


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [39]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))
salario_dict = dict(zip(datos_actual['trimestre'], datos_actual['salario básico unificado']))
ano = 2006

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [40]:
resultados_list = []

# Para cada trimeste
for t in [1, 2, 3, 4]:
    col_ingr = f'ingr_t_t{t}'
    umbral = umbral_dict.get(t)
    salario = salario_dict.get(t)
    
    # Selecciona el dataframe correspondiente
    if t == 1:
        df_actual = data3
    elif t == 2:
        df_actual = data6
    elif t == 3:
        df_actual = data9
    elif t == 4:
        df_actual = data12
    else:
        continue

    # Agrupa por región
    grouped = df_actual.groupby('region')
    
    for region_name, group in grouped:
        # 1. Filtra datos
        valid = group.dropna(subset=[col_ingr])
        
        if len(valid) == 0:
            continue
            
        # Extrae los vectores 
        ingresos = valid[col_ingr].values
        pesos = valid['fexp'].values
        
        # 2. Calcula indices
        gaps = (umbral - ingresos) / umbral
        gaps = np.clip(gaps, a_min=0, a_max=None)
        
        # 3. Calcula FGT
        total_poblacion = pesos.sum()
        
        # FGT0
        fgt0 = (pesos * (gaps > 0).astype(int)).sum() / total_poblacion
        
        # FGT1
        fgt1 = (pesos * (gaps ** 1)).sum() / total_poblacion
        
        # FGT2
        fgt2 = (pesos * (gaps ** 2)).sum() / total_poblacion
        
        # 4. Calcula Ingreso promedio
        ingreso_promedio = np.average(ingresos, weights=pesos)

        # 5. Desigualdad de Atkinson
        atkinson_resultados = {}
        
        if ingreso_promedio > 0:
            for epsilon in [0.25, 0.5, 0.75]:
                # La suma ponderada de la utilidad
                utility_sum = np.sum((ingresos ** (1 - epsilon)) * pesos)
                
                # promedio de esa utilidad
                utility_mean = utility_sum / total_poblacion
                
                # ingreso equivalente
                y_ede = utility_mean ** (1 / (1 - epsilon))
                
                # índice final
                atkinson_index = 1 - (y_ede / ingreso_promedio)
                atkinson_resultados[f'a{int(epsilon*100)}'] = atkinson_index
        else:
            # Si nadie gana nada, definimos desigualdad como NaN
            atkinson_resultados = {'a25': np.nan, 'a50': np.nan, 'a75': np.nan}

        # 6. Calcula mediana del ingreso
        # Ordena
        sort_idx = np.argsort(ingresos)
        ingreso_ordenado = ingresos[sort_idx]
        pesos_ordenado = pesos[sort_idx]
        cumsum_pesos = np.cumsum(pesos_ordenado)
        cutoff = total_poblacion / 2.0
        mediana = ingreso_ordenado[np.searchsorted(cumsum_pesos, cutoff)]
        
        # 7. Guarda resultados
        resultados_list.append({
            'ano': ano,
            'trimestre': t,
            'region': region_name,
            'fgt0': fgt0,
            'fgt1': fgt1,
            'fgt2': fgt2,
            'a25': atkinson_resultados['a25'],
            'a50': atkinson_resultados['a50'],
            'a75': atkinson_resultados['a75'],
            'ingreso_promedio': ingreso_promedio,
            'ingreso_mediana': mediana + 1 if mediana < 1 else mediana,
            'salario_minimo': salario,
            'kaitz_indice': salario / (mediana + 1 if mediana < 1 else mediana)         
        })

# lista a DataFrame
df_final_regional = pd.DataFrame(resultados_list)
df_final_regional

,ano,trimestre,region,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio,ingreso_mediana,salario_minimo,kaitz_indice
0,2006,1,Amazonía,0.262505,0.096929,0.052884,0.053842,0.105750,0.157223,135.971997,108.817214,160.0,1.470356
1,2006,1,Azuay,0.247630,0.102424,0.058468,0.060424,0.120541,0.180856,158.325914,135.723465,160.0,1.178868
2,2006,1,"Costa, Santo Domingo",0.325752,0.152814,0.081594,0.061281,0.121956,0.182059,129.067271,113.293667,160.0,1.412259
3,2006,1,El Oro,0.346716,0.137578,0.079833,0.062087,0.121339,0.178499,118.772717,92.237361,160.0,1.734655
4,2006,1,Guayas,0.356618,0.139692,0.077968,0.069967,0.135707,0.197970,129.259098,98.382115,160.0,1.626312
5,2006,1,Los Ríos,0.434186,0.172837,0.095763,0.060019,0.119963,0.179661,115.432786,105.463468,160.0,1.517113
6,2006,1,Manabí,0.655179,0.294387,0.177357,0.061477,0.123379,0.185815,74.949718,62.264778,160.0,2.569671
7,2006,1,Pichincha,0.211752,0.077032,0.036494,0.067586,0.130584,0.190979,180.256369,139.669292,160.0,1.145563
8,2006,1,Sierra,0.358647,0.146923,0.076620,0.069088,0.136110,0.203109,136.442602,108.155353,160.0,1.479353
9,2006,2,Amazonía,0.344910,0.123315,0.059013,0.060736,0.120641,0.182847,141.290005,110.772322,160.0,1.444404


### Inserta los cálculos en la base final

In [41]:
indices = pd.read_csv("indices_region.csv", encoding='latin-1')

In [42]:
import os

# 2. cheque el archivo
if not os.path.isfile('indices_region.csv'):
    # Headers si es la primera vez
    df_final_regional.to_csv('indices_region.csv', index=False, encoding='latin-1')
else:
    # SI ya existe, append
    df_final_regional.to_csv('indices_region.csv', mode='a', index=False, header=False, encoding='latin-1')